In [ ]:
# Core data manipulationimport pandas as pdimport numpy as npimport osimport warningswarnings.filterwarnings('ignore')# Text processingimport reimport nltkfrom nltk.corpus import stopwordsfrom nltk.tokenize import word_tokenizefrom nltk.stem import WordNetLemmatizer# Audio processingimport librosaimport soundfile as sf# Machine Learningimport tensorflow as tffrom tensorflow import kerasfrom sklearn.model_selection import (    train_test_split,     cross_val_score,     StratifiedKFold)from sklearn.preprocessing import (    StandardScaler,     LabelEncoder,    OneHotEncoder)from sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipeline# Machine Learning Modelsfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.svm import SVCfrom sklearn.naive_bayes import MultinomialNBfrom sklearn.linear_model import LogisticRegression# Evaluation Metricsfrom sklearn.metrics import (    accuracy_score,    precision_score,    recall_score,    f1_score,    confusion_matrix,    classification_report,    roc_auc_score)# Visualizationimport matplotlib.pyplot as pltimport seaborn as sns# Set random seedsnp.random.seed(42)tf.random.set_seed(42)# Download NLTK resourcesnltk.download('punkt', quiet=True)nltk.download('stopwords', quiet=True)nltk.download('wordnet', quiet=True)print("Environment setup complete.")

In [ ]:
class MultimodalDataProcessor:    def __init__(self, base_path):        """        Initialize multimodal data processor                Args:            base_path (str): Base directory for dataset        """        self.base_path = base_path        self.paths = {            'csv': os.path.join(base_path, 'overview-of-recordings.csv'),            'test_audio': os.path.join(base_path, 'test'),            'train_audio': os.path.join(base_path, 'train'),            'validate_audio': os.path.join(base_path, 'validate')        }        def load_dataset(self):        """        Load dataset metadata                Returns:            pd.DataFrame: Loaded dataset        """        try:            df = pd.read_csv(self.paths['csv'])            df.dropna(subset=['Phrase', 'Filename', 'Prompt'], inplace=True)                        print("Dataset Loaded Successfully!")            print(f"Dataset Shape: {df.shape}")            print(f"Unique Prompts: {df['Prompt'].nunique()}")                        return df        except Exception as e:            print(f"Error loading dataset: {e}")            return None        def preprocess_text(self, text):        """        Comprehensive text preprocessing                Args:            text (str): Input text                Returns:            str: Preprocessed text        """        # Convert to lowercase        text = str(text).lower()                # Remove special characters        text = re.sub(r'[^a-zA-Z\s]', '', text)                # Tokenization        tokens = word_tokenize(text)                # Remove stopwords        stop_words = set(stopwords.words('english'))        tokens = [token for token in tokens if token not in stop_words]                # Lemmatization        lemmatizer = WordNetLemmatizer()        tokens = [lemmatizer.lemmatize(token) for token in tokens]                return ' '.join(tokens)        def extract_audio_features(self, audio_path, sample_rate=16000):        """        Extract comprehensive audio features                Args:            audio_path (str): Path to audio file            sample_rate (int): Sampling rate                Returns:            numpy.ndarray: Extracted features        """        try:            # Load audio            audio, sr = librosa.load(audio_path, sr=sample_rate)                        # Feature extraction            features = []                        # MFCCs            mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)            features.extend(np.mean(mfccs.T, axis=0))            features.extend(np.std(mfccs.T, axis=0))                        # Spectral Centroid            spectral_centroids = librosa.feature.spectral_centroid(y=audio, sr=sr)            features.append(np.mean(spectral_centroids))                        # Chroma Features            chroma = librosa.feature.chroma_stft(y=audio, sr=sr)            features.extend(np.mean(chroma.T, axis=0))                        return np.array(features)                except Exception as e:            print(f"Audio feature extraction error: {e}")            return None        def prepare_multimodal_data(self, df):        """        Prepare multimodal dataset                Args:            df (pd.DataFrame): Input dataset                Returns:            tuple: Prepared features and labels        """        # Preprocess text        df['processed_text'] = df['Phrase'].apply(self.preprocess_text)                # Extract audio features        audio_features = []        audio_labels = []                for _, row in df.iterrows():            # Construct full audio file path            audio_path = os.path.join(                self.paths[f"{row['Dataset']}_audio"],                 row['Filename']            )                        # Extract audio features            audio_feat = self.extract_audio_features(audio_path)                        if audio_feat is not None:                audio_features.append(audio_feat)                audio_labels.append(row['Prompt'])                return {            'text': df['processed_text'],            'audio': np.array(audio_features),            'labels': np.array(audio_labels)        }# Initialize and process dataprocessor = MultimodalDataProcessor(    r"G:\Msc\NCU\Doctoral Record\multimodal_medical_diagnosis\data\recordings")df = processor.load_dataset()multimodal_data = processor.prepare_multimodal_data(df)

In [ ]:
class MultimodalFeaturePreparation:    def __init__(self, multimodal_data):        """        Prepare features for multimodal classification                Args:            multimodal_data (dict): Multimodal dataset        """        self.text = multimodal_data['text']        self.audio = multimodal_data['audio']        self.labels = multimodal_data['labels']        def prepare_features(self):        """        Prepare text and audio features                Returns:            tuple: Prepared features and labels        """        # Text vectorization        text_vectorizer = TfidfVectorizer(max_features=5000)        text_features = text_vectorizer.fit_transform(self.text)                # Audio feature scaling        audio_scaler = StandardScaler()        audio_features = audio_scaler.fit_transform(self.audio)                # Encode labels        label_encoder = LabelEncoder()        encoded_labels = label_encoder.fit_transform(self.labels)                # Split data        (            text_train, text_test,             audio_train, audio_test,             labels_train, labels_test        ) = train_test_split(            text_features, audio_features,             encoded_labels,             test_size=0.2,             random_state=42,             stratify=encoded_labels        )                return {            'text_train': text_train,            'text_test': text_test,            'audio_train': audio_train,            'audio_test': audio_test,            'labels_train': labels_train,            'labels_test': labels_test,            'label_encoder': label_encoder        }# Prepare featuresfeature_prep = MultimodalFeaturePreparation(multimodal_data)features = feature_prep.prepare_features()

In [ ]:
class MultimodalClassificationModels:    def __init__(self, features):        """        Initialize multimodal classification models                Args:            features (dict): Prepared features        """        self.text_train = features['text_train']        self.text_test = features['text_test']        self.audio_train = features['audio_train']        self.audio_test = features['audio_test']        self.labels_train = features['labels_train']        self.labels_test = features['labels_test']        self.label_encoder = features['label_encoder']        def create_text_models(self):        """        Create text classification models                Returns:            dict: Text classification models        """        text_models = {            'Logistic Regression': LogisticRegression(max_iter=1000),            'Naive Bayes': MultinomialNB(),            'SVM': SVC(probability=True)        }                text_results = {}                for name, model in text_models.items():            # Train model            model.fit(self.text_train, self.labels_train)                        # Predict            y_pred = model.predict(self.text_test)                        # Evaluate            text_results[name] = {                'Accuracy': accuracy_score(self.labels_test, y_pred),                'Precision': precision_score(self.labels_test, y_pred, average='weighted'),                'Recall': recall_score(self.labels_test, y_pred, average='weighted'),                'F1 Score': f1_score(self.labels_test, y_pred, average='weighted')            }                return text_results        def create_audio_models(self):        """        Create audio classification models                Returns:            dict: Audio classification models        """        audio_models = {            'Random Forest': RandomForestClassifier(),            'SVM': SVC(probability=True),            'Logistic Regression': LogisticRegression(max_iter=1000)        }                audio_results = {}                for name, model in audio_models.items():            # Train model            model.fit(self.audio_train, self.labels_train)                        # Predict            y_pred = model.predict(self.audio_test)                        # Evaluate            audio_results[name] = {                'Accuracy': accuracy_score(self.labels_test, y_pred),                'Precision': precision_score(self.labels_test, y_pred, average='weighted'),                'Recall': recall_score(self.labels_test, y_pred, average='weighted'),                'F1 Score': f1_score(self.labels_test, y_pred, average='weighted')            }                return audio_results        def create_deep_learning_model(self):        """        Create deep learning multimodal model                Returns:            dict: Deep learning model results        """        # Prepare input layers        text_input = keras.layers.Input(shape=(self.text_train.shape[1],))        audio_input = keras.layers.Input(shape=(self.audio_train.shape[1],))                # Text feature processing        text_dense = keras.layers.Dense(64, activation='relu')(text_input)        text_dropout = keras.layers.Dropout(0.3)(text_dense)                # Audio feature processing        audio_dense = keras.layers.Dense(64, activation='relu')(audio_input)        audio_dropout = keras.layers.Dropout(0.3)(audio_dense)                # Concatenate features        merged = keras.layers.Concatenate()([text_dropout, audio_dropout])                # Classification layers        dense1 = keras.layers.Dense(128, activation='relu')(merged)        dropout1 = keras.layers.Dropout(0.4)(dense1)        dense2 = keras.layers.Dense(64, activation='relu')(dropout1)        dropout2 = keras.layers.Dropout(0.3)(dense2)                # Output layer        output = keras.layers.Dense(            len(np.unique(self.labels_train)),             activation='softmax'        )(dropout2)                # Compile model        model = keras.Model(            inputs=[text_input, audio_input],             outputs=output        )        model.compile(            optimizer='adam',            loss='sparse_categorical_crossentropy',            metrics=['accuracy']        )                # Train model        model.fit(            [self.text_train.toarray(), self.audio_train],             self.labels_train,            epochs=50,            batch_size=32,            validation_split=0.2,            verbose=0        )                # Evaluate model        y_pred = model.predict([self.text_test.toarray(), self.audio_test])        y_pred_classes = np.argmax(y_pred, axis=1)                return {            'Multimodal Deep Learning': {                'Accuracy': accuracy_score(self.labels_test, y_pred_classes),                'Precision': precision_score(self.labels_test, y_pred_classes, average='weighted'),                'Recall': recall_score(self.labels_test, y_pred_classes, average='weighted'),                'F1 Score': f1_score(self.labels_test, y_pred_classes, average='weighted')            }        }# Run classificationsclassifier = MultimodalClassificationModels(features)text_results = classifier.create_text_models()audio_results = classifier.create_audio_models()multimodal_results = classifier.create_deep_learning_model()# Combine resultsall_results = {**text_results, **audio_results, **multimodal_results}

In [ ]:
def visualize_results(results):    """    Visualize classification results        Args:        results (dict): Classification results    """    # Prepare data for plotting    metrics_df = pd.DataFrame.from_dict(results, orient='index')        # Plot    plt.figure(figsize=(14, 7))        # Accuracy subplot    plt.subplot(2, 2, 1)    metrics_df['Accuracy'].plot(kind='bar')    plt.title('Model Accuracy')    plt.xlabel('Models')    plt.ylabel('Accuracy')    plt.xticks(rotation=45)        # Precision subplot    plt.subplot(2, 2, 2)    metrics_df['Precision'].plot(kind='bar')    plt.title('Model Precision')    plt.xlabel('Models')    plt.ylabel('Precision')    plt.xticks(rotation=45)        # Recall subplot    plt.subplot(2, 2, 3)    metrics_df['Recall'].plot(kind='bar')    plt.title('Model Recall')    plt.xlabel('Models')    plt.ylabel('Recall')    plt.xticks(rotation=45)        # F1 Score subplot    plt.subplot(2, 2, 4)    metrics_df['F1 Score'].plot(kind='bar')    plt.title('Model F1 Score')    plt.xlabel('Models')    plt.ylabel('F1 Score')    plt.xticks(rotation=45)        plt.tight_layout()    plt.show()def validate_hypotheses(results):    """    Validate research hypotheses        Args:        results (dict): Classification results    """    # Hypothesis validation threshold    threshold = 0.75        print("\n--- Research Hypothesis Validation ---")        # Text Classification Hypothesis    text_models = {k: v for k, v in results.items()                    if 'Text' not in k and 'Audio' not in k and 'Multimodal' not in k}    text_hypothesis_met = any(        metrics['Precision'] > threshold and         metrics['Recall'] > threshold        for metrics in text_models.values()    )        # Audio Classification Hypothesis    audio_models = {k: v for k, v in results.items()                     if 'Audio' in k or 'Multimodal' in k}    audio_hypothesis_met = any(        metrics['Precision'] > threshold and         metrics['Recall'] > threshold        for metrics in audio_models.values()    )        # Print validation results    print("\nText Classification Hypothesis:")    print("H10/H1a Hypothesis" +           (" SUPPORTED" if text_hypothesis_met else " NOT SUPPORTED"))        print("\nAudio Classification Hypothesis:")    print("H20/H2a Hypothesis" +           (" SUPPORTED" if audio_hypothesis_met else " NOT SUPPORTED"))# Visualize and validate resultsvisualize_results(all_results)validate_hypotheses(all_results)# Print detailed resultsfor model, metrics in all_results.items():    print(f"\n{model} Performance:")    for metric, value in metrics.items():        print(f"{metric}: {value:.4f}")

In [ ]:
def main():    """    Main execution function for multimodal classification    """    # Process data    processor = MultimodalDataProcessor(        r"G:\Msc\NCU\Doctoral Record\multimodal_medical_diagnosis\data\recordings"    )    df = processor.load_dataset()    multimodal_data = processor.prepare_multimodal_data(df)        # Prepare features    feature_prep = MultimodalFeaturePreparation(multimodal_data)    features = feature_prep.prepare_features()        # Run classifications    classifier = MultimodalClassificationModels(features)    text_results = classifier.create_text_models()    audio_results = classifier.create_audio_models()    multimodal_results = classifier.create_deep_learning_model()        # Combine and analyze results    all_results = {**text_results, **audio_results, **multimodal_results}        # Visualize and validate    visualize_results(all_results)    validate_hypotheses(all_results)# Run main executionif __name__ == '__main__':    main()